# 08 - Robustesse inter-seed

Je réentraîne le BNN image-only (torchbnn) et le MC Dropout (taux 0.3) sur plusieurs seeds pour vérifier que les conclusions principales du mémoire ne dépendent pas d'une seule initialisation aléatoire. Je regarde surtout le σ médian in-distribution, qui est l'argument central en faveur de torchbnn, ainsi que l'AUC et l'ECE.

Les valeurs de la seed 42 sont écrites en dur (elles correspondent à celles déjà rapportées dans le mémoire) pour ne pas avoir à tout réentraîner. Les autres seeds sont calculées par ce notebook.

### 1. Imports et configuration

In [2]:
import os, glob, random, pickle
import numpy as np
import torch
import torch.nn as nn
import torchbnn as bnn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
from PIL import Image
import matplotlib.pyplot as plt

device = torch.device("mps" if torch.backends.mps.is_available() else
                      "cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

DATAROOT  = "/Users/teodul/Documents/Memoire/Brain Tumor MRI clean"
TRAIN_DIR = os.path.join(DATAROOT, "Train")
TEST_DIR  = os.path.join(DATAROOT, "Test")
CNN_CKPT  = "runs_cnn_baseline/best_final.pt"
SAVE_DIR  = "runs_seed_robustness"
os.makedirs(SAVE_DIR, exist_ok=True)

IMG_SIZE     = 224
BATCH_SIZE   = 32
EPOCHS       = 15
LR           = 1e-3
DROPOUT_RATE = 0.3
MC_PASSES    = 50
KL_WEIGHT    = 0.01
PATIENCE     = 3
M_BINS       = 10
CLASSES      = ["glioma", "meningioma", "notumor", "pituitary"]
NUM_CLASSES  = 4

SEED_REF      = 42
SEEDS_TO_RUN  = [0, 1, 2, 3]   # seeds supplémentaires à tester

with open("predicted_clinical.pkl", "rb") as f:
    predicted_clinical = pickle.load(f)

device: mps


### 2. Dataset et transforms

In [3]:
class MultimodalBrainDataset(Dataset):
    def __init__(self, rootdir, transform, split=None, classes=CLASSES, samples=None):
        self.transform = transform
        self.classes   = classes
        if samples is not None:
            self.samples = list(samples)
        else:
            self.samples = []
            for idx, cls in enumerate(classes):
                clsdir = os.path.join(rootdir, cls)
                for ext in ['.jpg', '.jpeg', '.png']:
                    for path in sorted(glob.glob(os.path.join(clsdir, f'*{ext}'))):
                        clin = predicted_clinical[split].get(path, np.array([0.5], dtype=np.float32))
                        self.samples.append((path, idx, cls, clin))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label, clsname, clinical = self.samples[idx]
        img = self.transform(Image.open(path).convert('RGB'))
        clinical = torch.tensor(clinical, dtype=torch.float32)
        return img, clinical, label


train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])


def build_loaders(seed):
    all_samples = []
    for idx, cls in enumerate(CLASSES):
        clsdir = os.path.join(TRAIN_DIR, cls)
        for ext in ['.jpg', '.jpeg', '.png']:
            for path in sorted(glob.glob(os.path.join(clsdir, f'*{ext}'))):
                clin = predicted_clinical["train"].get(path, np.array([0.5], dtype=np.float32))
                all_samples.append((path, idx, cls, clin))

    train_samples, val_samples = [], []
    rng = np.random.default_rng(seed)
    for cls_idx, cls in enumerate(CLASSES):
        cls_samps = [(p, l, c, cl) for p, l, c, cl in all_samples if l == cls_idx]
        idxs      = list(rng.permutation(len(cls_samps)))
        n_val     = max(1, int(0.15 * len(cls_samps)))
        val_samples   += [cls_samps[i] for i in idxs[:n_val]]
        train_samples += [cls_samps[i] for i in idxs[n_val:]]

    random.seed(seed); random.shuffle(train_samples)

    train_ds = MultimodalBrainDataset(None, train_tf, samples=train_samples)
    val_ds   = MultimodalBrainDataset(None, val_tf,   samples=val_samples)
    test_ds  = MultimodalBrainDataset(TEST_DIR, val_tf, split="test")

    labels         = [s[1] for s in train_ds.samples]
    counts         = np.bincount(labels)
    class_weights  = 1.0 / counts
    sample_weights = class_weights[labels]
    g_gen = torch.Generator(); g_gen.manual_seed(seed)
    sampler = WeightedRandomSampler(
        weights=torch.tensor(sample_weights, dtype=torch.double),
        num_samples=len(sample_weights), replacement=True)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                              num_workers=0, generator=g_gen)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    return train_ds, train_loader, val_loader, test_loader

### 3. Architectures

In [4]:
class BNNImageOnly(nn.Module):
    def __init__(self, num_classes=4, cnn_ckpt=None, prior_mu=0, prior_sigma=0.1):
        super().__init__()
        backbone    = models.resnet18(weights="IMAGENET1K_V1")
        backbone.fc = nn.Identity()
        self.cnn    = backbone
        if cnn_ckpt and os.path.exists(cnn_ckpt):
            ckpt     = torch.load(cnn_ckpt, map_location="cpu", weights_only=False)
            state    = ckpt.get("model_state", ckpt)
            filtered = {k: v for k, v in state.items() if not k.startswith("fc.")}
            self.cnn.load_state_dict(filtered, strict=False)
        for name, param in self.cnn.named_parameters():
            param.requires_grad = ("layer4" in name or "layer3" in name)
        self.bnn_head = nn.Sequential(
            bnn.BayesLinear(prior_mu=prior_mu, prior_sigma=prior_sigma,
                            in_features=512, out_features=256), nn.ReLU(),
            bnn.BayesLinear(prior_mu=prior_mu, prior_sigma=prior_sigma,
                            in_features=256, out_features=128), nn.ReLU(),
            bnn.BayesLinear(prior_mu=prior_mu, prior_sigma=prior_sigma,
                            in_features=128, out_features=num_classes),
        )
    def forward(self, img, clinical=None):
        return self.bnn_head(self.cnn(img))


class MCDropoutImageOnly(nn.Module):
    def __init__(self, num_classes=4, dropout=0.3):
        super().__init__()
        backbone = models.resnet18(weights="IMAGENET1K_V1")
        backbone.fc = nn.Identity()
        self.cnn = backbone
        for name, param in self.cnn.named_parameters():
            param.requires_grad = ("layer4" in name or "layer3" in name)
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(512, 256), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )
    def forward(self, img, clin=None):
        return self.head(self.cnn(img))
    def enable_mc_dropout(self):
        for m in self.modules():
            if isinstance(m, nn.Dropout):
                m.train()


def kl_loss(model):
    kl = torch.tensor(0.0, device=device)
    for m in model.modules():
        if isinstance(m, bnn.BayesLinear):
            mu        = m.weight_mu
            log_sigma = m.weight_log_sigma
            sigma     = torch.exp(log_sigma)
            prior_mu  = torch.tensor(m.prior_mu, device=device)
            prior_sig = torch.tensor(m.prior_sigma, device=device)
            kl += 0.5 * torch.mean(
                ((mu - prior_mu) ** 2 + sigma ** 2) / (prior_sig ** 2)
                - 1 + 2 * torch.log(prior_sig) - 2 * log_sigma)
    return kl

### 4. Métriques : AUC val, σ médian in-dist, ECE

In [5]:
@torch.no_grad()
def mc_inference(loader, model, n_passes=MC_PASSES, is_dropout=False):
    model.eval()
    if is_dropout:
        model.enable_mc_dropout()
    all_y, all_mean, all_std = [], [], []
    for img, clin, y in loader:
        img, clin = img.to(device), clin.to(device)
        passes = torch.stack([
            torch.softmax(model(img, clin), dim=1) for _ in range(n_passes)
        ])
        all_y.extend(y.numpy())
        all_mean.extend(passes.mean(dim=0).cpu().numpy())
        all_std.extend(passes.std(dim=0).cpu().numpy())
    all_y    = np.array(all_y)
    all_mean = np.array(all_mean)
    all_std  = np.array(all_std)
    preds    = all_mean.argmax(axis=1)
    # même définition que les notebooks 3, 6, 7 : sigma max sur les 4 classes
    sigma_in = all_std.max(axis=1)
    auc      = roc_auc_score(all_y, all_mean, multi_class="ovr", average="macro")
    bal_acc  = balanced_accuracy_score(all_y, preds)
    return all_y, all_mean, preds, sigma_in, auc, bal_acc


def compute_ece(all_mean, all_y, M=M_BINS):
    confs = all_mean.max(axis=1)
    preds = all_mean.argmax(axis=1)
    corrects = (preds == all_y).astype(float)
    bin_edges = np.linspace(0, 1, M + 1)
    ece = 0.0
    for i in range(M):
        lo, hi = bin_edges[i], bin_edges[i + 1]
        mask = (confs >= lo) & (confs < hi)
        if mask.sum() > 0:
            acc  = corrects[mask].mean()
            conf = confs[mask].mean()
            ece += (mask.sum() / len(confs)) * abs(acc - conf)
    return ece


def triage_metrics(preds, y, sigma):
    correct = (preds == y)
    sig_ok  = sigma[correct].mean()
    sig_err = sigma[~correct].mean()
    ratio   = float(sig_err / sig_ok) if (correct.sum() > 0 and sig_ok > 0) else float("nan")

    order      = np.argsort(sigma)              # sigma croissant : plus confiant d'abord
    corr_s     = correct[order].astype(float)
    n          = len(corr_s)
    acc_global = float(corr_s.mean())
    acc_top50  = float(corr_s[: n // 2].mean())

    err_s = (corr_s == 0).astype(float)         # erreurs, dans l'ordre sigma croissant
    n_err = err_s.sum()
    k20   = int(0.20 * n)
    err20 = float(err_s[n - k20:].sum() / n_err * 100) if n_err > 0 else float("nan")
    return ratio, acc_global, acc_top50, err20

### 5. Entraînement d'une seed (les deux modèles)

In [6]:
def train_bnn(seed, train_loader, val_loader):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    model = BNNImageOnly(cnn_ckpt=CNN_CKPT).to(device)
    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.Adam(trainable, lr=LR)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2)
    ce_loss   = nn.CrossEntropyLoss()
    best_auc, best_state = -1.0, None

    for epoch in range(1, EPOCHS + 1):
        model.train()
        for img, clin, y in train_loader:
            img, y = img.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(img)
            loss   = ce_loss(logits, y) + KL_WEIGHT * kl_loss(model)
            loss.backward(); optimizer.step()
        scheduler.step()
        _, mean_v, _, _, auc_v, _ = mc_inference(val_loader, model)
        if auc_v > best_auc:
            best_auc = auc_v
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model


def train_mcd(seed, train_loader, val_loader, dropout=DROPOUT_RATE):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    model = MCDropoutImageOnly(dropout=dropout).to(device)
    ckpt  = torch.load(CNN_CKPT, map_location="cpu", weights_only=False)
    state = ckpt.get("model_state", ckpt)
    filtered = {k: v for k, v in state.items() if not k.startswith("fc.")}
    model.cnn.load_state_dict(filtered, strict=False)
    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.Adam(trainable, lr=LR)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2)
    criterion = nn.CrossEntropyLoss()
    best_auc, best_state = -1.0, None

    for epoch in range(EPOCHS):
        model.train()
        for img, clin, y in train_loader:
            img, y = img.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(img), y)
            loss.backward(); optimizer.step()
        scheduler.step()
        model.eval()
        with torch.no_grad():
            ys, probs = [], []
            for img, clin, y in val_loader:
                img = img.to(device)
                probs.extend(torch.softmax(model(img), dim=1).cpu().numpy())
                ys.extend(y.numpy())
        auc_v = roc_auc_score(np.array(ys), np.array(probs),
                              multi_class="ovr", average="macro")
        if auc_v > best_auc:
            best_auc = auc_v
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model

### 6. Boucle sur les seeds

In [7]:
results = {
    "torchbnn": {
        42: {"auc": 0.9951, "ece": 0.0085, "sigma_med": 0.0217,
             "ratio": 4.83, "acc_global": 0.964, "acc_top50": 0.997, "err20": 87.8},
    },
    "mcd03": {
        42: {"auc": 0.998,  "ece": 0.0104, "sigma_med": 0.0010},
    },
}

for seed in SEEDS_TO_RUN:
    print(f"\n========== SEED {seed} ==========")
    _, train_loader, val_loader, test_loader = build_loaders(seed)

    print("  [torchbnn] entraînement...")
    m_bnn = train_bnn(seed, train_loader, val_loader)
    y, mean, preds, sigma_in, auc, bal = mc_inference(test_loader, m_bnn)
    ece = compute_ece(mean, y)
    ratio, acc_g, acc_50, err20 = triage_metrics(preds, y, sigma_in)
    results["torchbnn"][seed] = {"auc": auc, "ece": ece,
                                 "sigma_med": float(np.median(sigma_in)),
                                 "ratio": ratio, "acc_global": acc_g,
                                 "acc_top50": acc_50, "err20": err20}
    print(f"    AUC={auc:.4f}  ECE={ece:.4f}  sigma_med_in_dist={np.median(sigma_in):.4f}")
    print(f"    ratio={ratio:.2f}  acc_glob={acc_g:.3f}  acc_top50={acc_50:.3f}  err@20%={err20:.1f}%")

    print("  [MC Dropout 0.3] entraînement...")
    m_mcd = train_mcd(seed, train_loader, val_loader)
    y, mean, preds, sigma_in, auc, bal = mc_inference(test_loader, m_mcd, is_dropout=True)
    ece = compute_ece(mean, y)
    results["mcd03"][seed] = {"auc": auc, "ece": ece,
                              "sigma_med": float(np.median(sigma_in))}
    print(f"    AUC={auc:.4f}  ECE={ece:.4f}  sigma_med_in_dist={np.median(sigma_in):.4f}")


========== SEED 0 ==========
  [torchbnn] entraînement...
    AUC=0.9957  ECE=0.0068  sigma_med_in_dist=0.0158
    ratio=4.47  acc_glob=0.958  acc_top50=0.998  err@20%=85.4%
  [MC Dropout 0.3] entraînement...
    AUC=0.9989  ECE=0.0132  sigma_med_in_dist=0.0002

========== SEED 1 ==========
  [torchbnn] entraînement...
    AUC=0.9953  ECE=0.0134  sigma_med_in_dist=0.0277
    ratio=3.52  acc_glob=0.957  acc_top50=0.995  err@20%=83.7%
  [MC Dropout 0.3] entraînement...
    AUC=0.9978  ECE=0.0141  sigma_med_in_dist=0.0004

========== SEED 2 ==========
  [torchbnn] entraînement...
    AUC=0.9962  ECE=0.0122  sigma_med_in_dist=0.0302
    ratio=3.49  acc_glob=0.962  acc_top50=0.997  err@20%=86.4%
  [MC Dropout 0.3] entraînement...
    AUC=0.9987  ECE=0.0138  sigma_med_in_dist=0.0002

========== SEED 3 ==========
  [torchbnn] entraînement...
    AUC=0.9949  ECE=0.0240  sigma_med_in_dist=0.0613
    ratio=2.15  acc_glob=0.962  acc_top50=0.991  err@20%=72.7%
  [MC Dropout 0.3] entraînement...
 

### 7. Tableau récapitulatif et moyennes ± écarts-types

In [8]:
def summarize(model_key, label):
    seeds = sorted(results[model_key].keys())
    aucs   = np.array([results[model_key][s]["auc"]       for s in seeds])
    eces   = np.array([results[model_key][s]["ece"]       for s in seeds])
    sigmas = np.array([results[model_key][s]["sigma_med"] for s in seeds])
    print(f"\n{label}")
    print(f"  {'seed':>6} {'AUC':>8} {'ECE':>8} {'sigma_med':>10}")
    for s in seeds:
        r = results[model_key][s]
        print(f"  {s:>6} {r['auc']:>8.4f} {r['ece']:>8.4f} {r['sigma_med']:>10.4f}")
    print(f"  {'moy':>6} {aucs.mean():>8.4f} {eces.mean():>8.4f} {sigmas.mean():>10.4f}")
    print(f"  {'std':>6} {aucs.std():>8.4f} {eces.std():>8.4f} {sigmas.std():>10.4f}")
    return aucs, eces, sigmas


def summarize_triage(model_key="torchbnn"):
    seeds = sorted(results[model_key].keys())
    ratio = np.array([results[model_key][s]["ratio"]      for s in seeds])
    accg  = np.array([results[model_key][s]["acc_global"] for s in seeds])
    acc50 = np.array([results[model_key][s]["acc_top50"]  for s in seeds])
    err20 = np.array([results[model_key][s]["err20"]      for s in seeds])
    print("\n=== torchbnn : ratio et triage inter-seed ===")
    print(f"  {'seed':>6} {'ratio':>7} {'acc_glob':>9} {'acc_top50':>10} {'err@20%':>9}")
    for s in seeds:
        r = results[model_key][s]
        print(f"  {s:>6} {r['ratio']:>7.2f} {r['acc_global']:>9.3f} {r['acc_top50']:>10.3f} {r['err20']:>8.1f}%")
    print(f"  {'moy':>6} {ratio.mean():>7.2f} {accg.mean():>9.3f} {acc50.mean():>10.3f} {err20.mean():>8.1f}%")
    print(f"  {'std':>6} {ratio.std():>7.2f} {accg.std():>9.3f} {acc50.std():>10.3f} {err20.std():>8.1f}%")


summarize("torchbnn", "=== torchbnn (BNN image-only) ===")
summarize("mcd03",    "=== MC Dropout 0.3 ===")
summarize_triage()


=== torchbnn (BNN image-only) ===
    seed      AUC      ECE  sigma_med
       0   0.9957   0.0068     0.0158
       1   0.9953   0.0134     0.0277
       2   0.9962   0.0122     0.0302
       3   0.9949   0.0240     0.0613
      42   0.9951   0.0085     0.0217
     moy   0.9954   0.0130     0.0313
     std   0.0004   0.0060     0.0158

=== MC Dropout 0.3 ===
    seed      AUC      ECE  sigma_med
       0   0.9989   0.0132     0.0002
       1   0.9978   0.0141     0.0004
       2   0.9987   0.0138     0.0002
       3   0.9985   0.0161     0.0001
      42   0.9980   0.0104     0.0010
     moy   0.9984   0.0135     0.0004
     std   0.0004   0.0018     0.0003

=== torchbnn : ratio et triage inter-seed ===
    seed   ratio  acc_glob  acc_top50   err@20%
       0    4.47     0.958      0.998     85.4%
       1    3.52     0.957      0.995     83.7%
       2    3.49     0.962      0.997     86.4%
       3    2.15     0.962      0.991     72.7%
      42    4.83     0.964      0.997     87.8